# MASC + UPipe — Kaggle 2x T4

**Notebook tu dong tai du lieu + setup + train**

| Resource | Value |
|---|---|
| GPU | 2x T4 16 GB |
| Model weights | `FoundationVision/LlamaGen` (HuggingFace) |
| Token codes | `Efficient-Large-Model/imagenet-llamagen-cache` (HuggingFace) |

> Yeu cau: Accelerator = GPU T4 x 2, Internet = ON

In [1]:
# ============================================================
# CELL 1 — Setup: Clone repo + install dependencies
# ============================================================
import subprocess, os, sys, time

WORK_DIR    = '/kaggle/working'
MASC_DIR    = f'{WORK_DIR}/MASC'
LLAMA_DIR   = f'{WORK_DIR}/LlamaGen'
WEIGHTS_DIR = f'{WORK_DIR}/weights'
CODES_DIR   = f'{WORK_DIR}/codes'
CKPT_DIR    = f'{WORK_DIR}/checkpoints'

for d in [WEIGHTS_DIR, CODES_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

def run(cmd):
    print(f'>>> {cmd}')
    return subprocess.run(cmd, shell=True).returncode

# Clone / Update MASC+UPipe fork
if not os.path.exists(MASC_DIR):
    run(f'git clone https://github.com/nta2112/masc-upgrade-ann {MASC_DIR}')
else:
    run(f'git -C {MASC_DIR} fetch origin')
    run(f'git -C {MASC_DIR} reset --hard origin/main')
    print('MASC da cap nhat ban moi nhat tu GitHub')

# Clone LlamaGen backbone
if not os.path.exists(LLAMA_DIR):
    run(f'git clone https://github.com/FoundationVision/LlamaGen {LLAMA_DIR}')
else:
    print('LlamaGen already cloned')

for p in [MASC_DIR, LLAMA_DIR]:
    if p not in sys.path: sys.path.insert(0, p)

run('pip install -q pyyaml huggingface_hub webdataset accelerate')
print('\nCell 1 done')


>>> git clone https://github.com/nta2112/masc-upgrade-ann /kaggle/working/MASC


Cloning into '/kaggle/working/MASC'...


>>> git clone https://github.com/FoundationVision/LlamaGen /kaggle/working/LlamaGen


Cloning into '/kaggle/working/LlamaGen'...


>>> pip install -q pyyaml huggingface_hub webdataset accelerate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.0 MB/s eta 0:00:00

Cell 1 done


In [2]:
# ============================================================
# CELL 2 — Tai LlamaGen Weights tu HuggingFace (~1.7 GB)
# ============================================================
from huggingface_hub import hf_hub_download
import os

WEIGHTS_DIR = '/kaggle/working/weights'

for fname in ['vq_ds16_c2i.pt', 'c2i_L_256.pt']:
    dest = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(dest):
        print(f'{fname} da co ({os.path.getsize(dest)/1e6:.0f} MB), bo qua')
        continue
    print(f'Dang tai {fname}...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen',
                    filename=fname, local_dir=WEIGHTS_DIR)
    print(f'Da tai {fname} ({os.path.getsize(dest)/1e6:.0f} MB)')

print('\nCell 2 done — weights files:')
for f in os.listdir(WEIGHTS_DIR):
    print(f'  {f}: {os.path.getsize(os.path.join(WEIGHTS_DIR,f))/1e6:.0f} MB')

Dang tai vq_ds16_c2i.pt...


vq_ds16_c2i.pt:   0%|          | 0.00/288M [00:00<?, ?B/s]

Da tai vq_ds16_c2i.pt (288 MB)
Dang tai c2i_L_256.pt...


c2i_L_256.pt:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

Da tai c2i_L_256.pt (1372 MB)

Cell 2 done — weights files:
  c2i_L_256.pt: 1372 MB
  .cache: 0 MB
  vq_ds16_c2i.pt: 288 MB


In [3]:
# ============================================================
# CELL 3 — Tai + Convert ImageNet Token Codes (~7.2 GB Tar -> NPY Shards)
# ETA: download bo qua neu da co, convert ~8-12 phut (1.28M anh)
# ============================================================
import os, io, tarfile, time
import numpy as np
from pathlib import Path
from huggingface_hub import snapshot_download

CODES_DIR  = '/kaggle/working/codes'
WDS_DIR    = '/kaggle/working/wds_cache'
SHARD_SIZE = 512  # 512 anh moi shard (~500 KB/file)

os.makedirs(WDS_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)

# -- Step A: Kiem tra / Tai tar tu HuggingFace --
existing_tars = sorted(Path(WDS_DIR).glob('**/*.tar'))
if existing_tars:
    print(f'{len(existing_tars)} tar files da co, bo qua download')
else:
    print('Dang tai imagenet-llamagen-cache tu HuggingFace... (co the mat 20-30 phut lan dau)')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    existing_tars = sorted(Path(WDS_DIR).glob('**/*.tar'))
    print(f'Da tai {len(existing_tars)} tar files')

# -- Step B: Convert Tar (mar_cache_vq/*.npz) -> NPY shards --
existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
labels_path  = f'{CODES_DIR}/labels.npy'
skip_convert = False
if existing_npy and os.path.exists(labels_path):
    try:
        cur_labels = np.load(labels_path)
        if len(cur_labels) > 0 and len(existing_npy) > 0:
            skip_convert = True
    except Exception:
        skip_convert = False

if skip_convert:
    print(f'{len(existing_npy)} NPY shards hop le da co, bo qua convert')
else:
    # Synset -> Class ID mapping (0..999)
    try:
        from masc.imagenet_synsets import SYNSET_TO_ID
    except Exception:
        import urllib.request, json
        u = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
        d = json.loads(urllib.request.urlopen(u).read().decode())
        SYNSET_TO_ID = {d[str(i)][0]: i for i in range(1000)}

    tar_path = existing_tars[0]
    print(f'Dang convert: {tar_path.name} ({os.path.getsize(tar_path)/1e9:.2f} GB)...')
    t0 = time.time()
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'):
                continue
            parts = member.name.split('/')
            if len(parts) < 2:
                continue
            synset = parts[-2]  # mar_cache_vq/<synset>/<id>.JPEG.npz
            label = SYNSET_TO_ID.get(synset, 0)

            f = tf.extractfile(member)
            if f is None:
                continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception:
                continue

            if len(codes) != 256:
                continue

            shard_codes.append(codes)
            all_labels.append(label)
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy',
                        np.array(shard_codes, dtype=np.int32))
                shard_codes = []
                shard_idx += 1
                if shard_idx % 50 == 0:
                    elapsed = (time.time() - t0) / 60
                    speed = total_imgs / max(1, time.time() - t0)
                    print(f'  Shard {shard_idx:4d} | {total_imgs:7,d} anh | {elapsed:.1f} phut | {speed:.0f} img/s', flush=True)

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy',
                np.array(shard_codes, dtype=np.int32))
        shard_idx += 1

    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    print(f'Xong! {total_imgs:,} anh -> {shard_idx} shards ({(time.time()-t0)/60:.1f} phut)')

# -- Kiem tra ket qua --
npy_files = sorted(Path(CODES_DIR).glob('shard_*.npy'))
if not npy_files:
    raise RuntimeError('0 shards duoc tao ra! Kiem tra file tar trong ' + WDS_DIR)
labels = np.load(labels_path)
sample = np.load(npy_files[0])
print(f'\nKet qua: {len(npy_files)} shards, {len(labels):,} labels')
print(f'Shape shard dau: {sample.shape}  (512 images x 256 tokens)')
print(f'Token range: [{sample.min()}, {sample.max()}]  (mong doi trong 0-16383)')
print('Cell 3 done - San sang cho Cell 4!')


Dang tai imagenet-llamagen-cache tu HuggingFace... (co the mat 20-30 phut lan dau)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Da tai 1 tar files
Dang convert: imagenet_llamagen_cache.tar (7.22 GB)...
  Shard   50 |  25,600 anh | 0.1 phut | 4107 img/s
  Shard  100 |  51,200 anh | 0.2 phut | 4115 img/s
  Shard  150 |  76,800 anh | 0.3 phut | 4069 img/s
  Shard  200 | 102,400 anh | 0.4 phut | 4094 img/s
  Shard  250 | 128,000 anh | 0.5 phut | 4134 img/s
  Shard  300 | 153,600 anh | 0.6 phut | 4191 img/s
  Shard  350 | 179,200 anh | 0.7 phut | 4195 img/s
  Shard  400 | 204,800 anh | 0.8 phut | 4201 img/s
  Shard  450 | 230,400 anh | 0.9 phut | 4236 img/s
  Shard  500 | 256,000 anh | 1.0 phut | 4257 img/s
  Shard  550 | 281,600 anh | 1.1 phut | 4263 img/s
  Shard  600 | 307,200 anh | 1.2 phut | 4282 img/s
  Shard  650 | 332,800 anh | 1.3 phut | 4281 img/s
  Shard  700 | 358,400 anh | 1.4 phut | 4279 img/s
  Shard  750 | 384,000 anh | 1.5 phut | 4269 img/s
  Shard  800 | 409,600 anh | 1.6 phut | 4235 img/s
  Shard  850 | 435,200 anh | 1.7 phut | 4240 img/s
  Shard  900 | 460,800 anh | 1.8 phut | 4238 img/s
  Shard 

In [4]:
# ============================================================
# CELL 4 — Build MASC Mapping (k=4096, ~2 phut)
# ============================================================
import sys, os, time
import numpy as np, torch

for p in ['/kaggle/working/MASC', '/kaggle/working/LlamaGen']:
    if p not in sys.path: sys.path.insert(0, p)

MAPPING_PATH = '/kaggle/input/datasets/nta212/masc-file/masc_mapping_k4096.npz'
WEIGHTS_DIR  = '/kaggle/working/weights'

if os.path.exists(MAPPING_PATH):
    from masc import load_mapping
    mp = load_mapping(MAPPING_PATH)
    print(f'Mapping da co: N={mp.n} -> k={mp.k}')
else:
    print('Dang load VQ-VAE codebook...')
    ckpt = torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu')
    state = ckpt.get('model', ckpt.get('ema', ckpt))
    
    # Lay truc tiep ma tran codebook tu state_dict
    codebook = state['quantize.embedding.weight'].detach().cpu().numpy()
    print(f'Codebook shape: {codebook.shape}')  # (16384, 8)

    print('Dang build MASC tree k=4096 (~2 phut)...')
    t0 = time.time()
    from masc import build_masc_tree, save_mapping
    tree = build_masc_tree(codebook, k=4096)
    save_mapping(MAPPING_PATH, tree.mapping, tree.k)
    print(f'Xong {(time.time()-t0):.1f}s — N={tree.n} -> k={tree.k}')
    print(f'Luu tai: {MAPPING_PATH}')

print('Cell 4 done')


Mapping da co: N=16384 -> k=4096
Cell 4 done


In [5]:
# ============================================================
# CELL 5 — Tao LlamaGen Backbone Adapter
# ============================================================
import os

MASC_DIR    = '/kaggle/working/MASC'
WEIGHTS_DIR = '/kaggle/working/weights'
backbone_dir = f'{MASC_DIR}/backbones'
os.makedirs(backbone_dir, exist_ok=True)

with open(f'{backbone_dir}/__init__.py', 'w') as f:
    f.write('from .llamagen_adapter import build_backbone\n')

code = '''
import os, sys
sys.path.insert(0, "/kaggle/working/LlamaGen")
sys.path.insert(0, "/kaggle/working/MASC")
import torch, torch.nn as nn

WEIGHTS_DIR = "/kaggle/working/weights"

class LlamaGenAdapter:
    def __init__(self, model): self.m = model
    def get_token_embedding(self): return self.m.tok_embeddings
    def set_token_embedding(self, e): self.m.tok_embeddings = e
    def get_output_head(self): return self.m.output
    def set_output_head(self, h): self.m.output = h

def build_backbone(cfg):
    from autoregressive.models.gpt import GPT_models
    name_map = {"llamagen_b":"GPT-B","llamagen_l":"GPT-L",
                "llamagen_xl":"GPT-XL","llamagen_xxl":"GPT-XXL"}
    model = GPT_models[name_map[cfg["name"]]](
        vocab_size=16384, block_size=256, num_classes=1000,
        cls_token_num=1, model_type="c2i",
        resid_dropout_p=0.1, ffn_dropout_p=0.1, drop_path_rate=0.0,
    )
    pt = cfg.get("pretrained")
    if pt and os.path.exists(pt):
        ckpt = torch.load(pt, map_location="cpu")
        state = ckpt.get("model", ckpt)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"Loaded {pt} | missing={len(missing)} unexpected={len(unexpected)}")
    else:
        print(f"No pretrained at {pt}, random init")
    return model, LlamaGenAdapter(model)
'''

with open(f'{backbone_dir}/llamagen_adapter.py', 'w') as f:
    f.write(code)

print(f'Adapter viet xong: {backbone_dir}/llamagen_adapter.py')
print('Cell 5 done')

Adapter viet xong: /kaggle/working/MASC/backbones/llamagen_adapter.py
Cell 5 done


In [6]:
# ============================================================
# CELL 6 — Smoke Test: Kiem tra pipeline truoc khi train
# ============================================================
import sys, os, torch, numpy as np
for p in ['/kaggle/working/MASC', '/kaggle/working/LlamaGen']:
    if p not in sys.path: sys.path.insert(0, p)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} | {props.total_memory/1e9:.1f} GB')

# Load mapping (uu tien file da upload / san co)
from masc import load_mapping
mapping_candidates = [
    '/kaggle/input/datasets/nta212/masc-file/masc_mapping_k4096.npz',
    '/kaggle/working/masc_mapping_k4096.npz',
]
map_path = next((p for p in mapping_candidates if os.path.exists(p)), '/kaggle/working/masc_mapping_k4096.npz')
mp = load_mapping(map_path)
print(f'\nMASC mapping: N={mp.n} -> k={mp.k} ({map_path})')

# Load model + adapter
from backbones import build_backbone
model, adapter = build_backbone({'name': 'llamagen_l',
    'pretrained': '/kaggle/working/weights/c2i_L_256.pt', 'image_size': 256})
model = model.to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params/1e6:.0f}M params')

# Apply MASC + UPipe
from masc.integration import apply_masc_and_upipe, MASCObjective
registry = apply_masc_and_upipe(
    model=model, adapter=adapter, mapping=mp.mapping,
    k=mp.k, upipe_chunk_heads=4, upipe_use_flash=True)
print(f'UPipe: {len(registry)} layers wrapped')

# Forward pass
objective = MASCObjective(mp.mapping, device=device)
B, L = 2, 256
fine_codes   = torch.randint(0, mp.n, (B, L), device=device)
class_labels = torch.randint(0, 1000, (B,), device=device)
coarse_in    = objective.coarse_targets(fine_codes)

model.eval()
with torch.no_grad():
    amp_ctx = torch.amp.autocast('cuda', dtype=torch.float16) if torch.cuda.is_available() else torch.amp.autocast('cpu')
    with amp_ctx:
        out = model(coarse_in, class_labels)
        logits = out[0] if isinstance(out, tuple) else out
        if logits.shape[1] > coarse_in.shape[1]:
            logits = logits[:, :coarse_in.shape[1], :]
        loss = objective.loss(logits, fine_codes)

print(f'\nForward OK!')
print(f'  logits: {logits.shape} (expected [B={B}, L={L}, k={mp.k}])')
print(f'  loss  : {loss.item():.4f} (expected ~ln({mp.k})={np.log(mp.k):.2f})')
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated()/1e9
    print(f'  VRAM used: {used:.2f} GB')

model.train()
print('\nSMOKE TEST PASSED - San sang train!\n')


In [ ]:
# ============================================================
# CELL 7 — Launch Training (torchrun 2x T4)
# ETA: 15000 steps ~ 2.5-3 gio
# ============================================================
import subprocess, os, glob

MASC_DIR    = '/kaggle/working/MASC'
CODES_DIR   = '/kaggle/working/codes'
CKPT_DIR    = '/kaggle/working/checkpoints'
WEIGHTS_DIR = '/kaggle/working/weights'
mapping_candidates = [
    '/kaggle/input/datasets/nta212/masc-file/masc_mapping_k4096.npz',
    '/kaggle/working/masc_mapping_k4096.npz',
]
MAPPING = next((p for p in mapping_candidates if os.path.exists(p)), '/kaggle/working/masc_mapping_k4096.npz')

os.makedirs(CKPT_DIR, exist_ok=True)

env = os.environ.copy()
env['PYTHONPATH'] = f'{MASC_DIR}:{WEIGHTS_DIR}/../LlamaGen:' + env.get('PYTHONPATH', '')

# Auto-detect resume checkpoint
existing = sorted(glob.glob(f'{CKPT_DIR}/ckpt_*.pt'))
cmd = [
    'torchrun', '--nproc_per_node=2', '--master_port=29500',
    f'{MASC_DIR}/scripts/train.py',
    '--config',  f'{MASC_DIR}/configs/llamagen_l_masc_upipe_kaggle.yaml',
    '--mapping', MAPPING,
    '--codes',   CODES_DIR,
    '--out',     CKPT_DIR,
    '--upipe-chunk-heads', '4',
    '--grad-ckpt',
]

if existing:
    resume = existing[-1]
    print(f'Tiep tuc tu checkpoint: {resume}')
    cmd.extend(['--resume', resume])
else:
    print('Bat dau train tu pretrained backbone (chua co checkpoint truoc)')

print('Command:', ' '.join(cmd))
print('=' * 60)

proc = subprocess.Popen(cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\nExit code: {proc.returncode}')
print('DONE!' if proc.returncode == 0 else 'ERROR - xem log ben tren')


In [ ]:
# ============================================================
# CELL 8 — Monitor: VRAM + checkpoints
# (Chay cell nay trong luc Cell 7 dang chay o tab khac)
# ============================================================
import subprocess, os, glob, re, torch

print('=== GPU VRAM ===')
r = subprocess.run(['nvidia-smi',
    '--query-gpu=index,name,memory.used,memory.total,utilization.gpu',
    '--format=csv,noheader,nounits'], capture_output=True, text=True)
for line in r.stdout.strip().split('\n'):
    p = [x.strip() for x in line.split(',')]
    print(f'  GPU {p[0]} ({p[1]}): {p[2]}/{p[3]} MB | util {p[4]}%')

print('\n=== Checkpoints ===')
ckpts = sorted(glob.glob('/kaggle/working/checkpoints/ckpt_*.pt'))
print(f'{len(ckpts)} checkpoints saved')
for ck in ckpts[-3:]:
    step = re.search(r'ckpt_(\d+)', os.path.basename(ck))
    print(f'  {os.path.basename(ck)} | step={step.group(1) if step else "?"} '
          f'| {os.path.getsize(ck)/1e6:.0f} MB')

print('\n=== Disk ===')
r2 = subprocess.run('du -sh /kaggle/working/*', shell=True,
                    capture_output=True, text=True)
print(r2.stdout)